In [ ]:
import numpy as np

ROWS, COLS = 4, 4
NUM_STATES = ROWS * COLS
ACTIONS = ['up', 'down', 'left', 'right']
ACTION_PROB = 1.0
GAMMA = 0.9
THETA = 1e-6
NUM_ACTIONS = len(ACTIONS)
GOAL_STATE = 15

def state_to_pos(state):
    return state // COLS, state % COLS


def move(state, action):
    row, col = state_to_pos(state)
    
    if action == 'up':
        row = max(row - 1, 0)
    elif action == 'down':
        row = min(row + 1, ROWS - 1)
    elif action == 'left':
        col = max(col - 1, 0)
    elif action == 'right':
        col = min(col + 1, COLS - 1)
    
    new_state = row * COLS + col
    return new_state

def reward(state, action, next_state):
    return 0 if next_state == GOAL_STATE else -1


def value_iteration():
    V = np.zeros(NUM_STATES)
    
    while True:
        delta = 0
        for s in range(NUM_STATES):
            if s == GOAL_STATE:
                continue
            v = V[s]
            # به‌روزرسانی با استفاده از max over actions
            best_value = float('-inf')
            for a in ACTIONS:
                s_next = move(s, a)
                r = reward(s, a, s_next)
                value = r + GAMMA * V[s_next]
                best_value = max(best_value, value)
            V[s] = best_value
            delta = max(delta, abs(v - V[s]))
        if delta < THETA:
            break

    policy = np.empty(NUM_STATES, dtype=object)
    for s in range(NUM_STATES):
        if s == GOAL_STATE:
            policy[s] = 'goal'
            continue
        best_value = float('-inf')
        best_action = None
        for a in ACTIONS:
            s_next = move(s, a)
            r = reward(s, a, s_next)
            value = r + GAMMA * V[s_next]
            if value > best_value:
                best_value = value
                best_action = a
        policy[s] = best_action

    return V, policy

In [ ]:
def policy_iteration():
    policy = np.random.choice(ACTIONS, size=NUM_STATES)
    V = np.zeros(NUM_STATES)
    
    def policy_evaluation():
        while True:
            delta = 0
            for s in range(NUM_STATES):
                if s == GOAL_STATE:
                    continue
                v = V[s]
                action = policy[s]
                s_next = move(s, action)
                r = reward(s, action, s_next)
                V[s] = r + GAMMA * V[s_next]
                delta = max(delta, abs(v - V[s]))
            if delta < THETA:
                break

    def policy_improvement():
        is_stable = True
        for s in range(NUM_STATES):
            if s == GOAL_STATE:
                continue
            old_action = policy[s]
            best_value = float('-inf')
            best_action = None
            for a in ACTIONS:
                s_next = move(s, a)
                r = reward(s, a, s_next)
                value = r + GAMMA * V[s_next]
                if value > best_value:
                    best_value = value
                    best_action = a
            policy[s] = best_action
            if old_action != best_action:
                is_stable = False
        return is_stable

    iteration = 0
    while True:
        print(f"Policy Iteration {iteration}")
        policy_evaluation()
        is_stable = policy_improvement()
        if is_stable:
            break
        iteration += 1

    return V, policy

In [ ]:
def q_value_iteration():
    Q = np.zeros((NUM_STATES, NUM_ACTIONS))
    
    iteration = 0
    while True:
        delta = 0
        for s in range(NUM_STATES):
            if s == GOAL_STATE:
                continue
            q_old = Q[s].copy()
            for a_idx, a in enumerate(ACTIONS):
                s_next = move(s, a)
                r = reward(s, a, s_next)
                # max over a' of Q(s', a')
                max_q_next = np.max(Q[s_next]) if s_next != GOAL_STATE else 0
                Q[s, a_idx] = r + GAMMA * max_q_next
            delta = max(delta, np.max(np.abs(q_old - Q[s])))
        
        iteration += 1
        if delta < THETA:
            break

    policy = np.empty(NUM_STATES, dtype=object)
    for s in range(NUM_STATES):
        if s == GOAL_STATE:
            policy[s] = 'G'
        else:
            best_action_idx = np.argmax(Q[s])
            policy[s] = ACTIONS[best_action_idx]
    
    V = np.max(Q, axis=1)
    return V, policy, Q